# run_all_0416b_nb11 — fixed seller × adaptive buyer

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **all 6 fixed sellers**
* Buyer strategies covered  : **6 adaptive buyer profiles (2-stage and 3-stage)**
* Mode                      : **fixed seller × adaptive buyer**
* Target per-notebook runtime : **~19 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 11000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb11"


## EXPERIMENTS list — notebook 11

In [2]:
# 6 adaptive-buyer profiles × 6 fixed sellers × 7 runs = 252 runs (~18.9 h).

def _ab(name, subs, sched):
    return AdaptiveBuyer(substrats=subs, schedule=sched, label=name)

ADAPTIVE_BUYERS = [
    _ab("ab_anchor_to_concession",
        [AnchorDragBuyer(), ConcessionBuyer()],
        [8]),
    _ab("ab_wild_to_boulware",
        [WildBuyer(), BoulwareBuyer()],
        [8]),
    _ab("ab_budget_to_tft",
        [BudgetBuyer(), TitForTatBuyer()],
        [8]),
    _ab("ab_persistent_tft_concession",
        [PersistentBuyer(0.30), TitForTatBuyer(), ConcessionBuyer()],
        [6, 11]),
    _ab("ab_anchor_conceder_concession",
        [AnchorDragBuyer(), ConcederBuyer(), ConcessionBuyer()],
        [6, 11]),
    _ab("ab_random_concession_tft",
        [RandomBuyer(), ConcessionBuyer(), TitForTatBuyer()],
        [6, 11]),
]

FIXED_SELLERS = ["anchor_high", "cooperative", "boulware_seller",
                 "conceder_seller", "tit_for_tat_matcher", "relational"]

N_RUNS_PER_PAIR = 7

EXPERIMENTS = []
for s in FIXED_SELLERS:
    sched = fixed_seller_schedule(s, N_ROUNDS)
    for ab in ADAPTIVE_BUYERS:
        tag = f"sel_{s}__buy_{ab.name}"
        EXPERIMENTS.append({
            "tag": tag, "buyer": ab, "seller_schedule": sched,
            "n_runs": N_RUNS_PER_PAIR,
        })

print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 36
Total runs      : 252


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/36]  sel_anchor_high__buy_ab_anchor_to_concession
  buyer   = ab_anchor_to_concession
  seller  = anchor_high  (adaptive=False)
  n_runs  = 7   n_rounds = 15

  --- Run 1/7  seed=16000  prod=Protein bar (60g) cost=$1.71 mkt=$6.75 int=$0.58 ---
  R 1 BUY $1.91 T0 [buy=anchor_drag]  SEL $6.50 [sel=anchor_high]  "Thank you for your interest. Given the high quality and stro..."
  R 2 BUY $2.13 T2 [buy=anchor_drag]  SEL $6.30 [sel=anchor_high]  "I appreciate your continued interest. While I understand you..."
  R 3 BUY $2.20 T6 [buy=anchor_drag]  SEL $6.10 [sel=anchor_high]  "Thank you for your continued interest. Our Protein bars are ..."
  R 4 BUY $2.37 T7 [buy=anchor_drag]  SEL $6.00 [sel=anchor_high]  "I understand you're seeking a competitive price, but our Pro..."
  R 5 BUY $2.61 T0 [buy=anchor_drag]  SEL $5.90 [sel=anchor_high]  "Thank you for your improved offer. While it's moving in a mo..."
  R 6 BUY $2.72 T5 [buy=anchor_drag]  SEL $5.80 [sel=anchor_high]  "Thank you for your 

## Inspection

In [4]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 5652: illegal multibyte sequence